1# **Core Mathematical Activation Functions & Derivatives**

In [13]:
import numpy as np

def sigmoid(Z):
    """
    Computes the sigmoid activation of Z and stores cache for backpropagation.
    """
    A = 1 / (1 + np.exp(-Z))
    cache = Z
    return A, cache

def relu(Z):
    """
    Computes the Rectified Linear Unit (ReLU) activation of Z.
    """
    A = np.maximum(0, Z)
    cache = Z
    return A, cache

def sigmoid_backward(dA, cache):
    """
    Computes the gradient of the cost with respect to Z for a sigmoid unit.
    """
    Z = cache
    s = 1 / (1 + np.exp(-Z))
    dZ = dA * s * (1 - s)
    return dZ

def relu_backward(dA, cache):
    """
    Computes the gradient of the cost with respect to Z for a ReLU unit.
    """
    Z = cache
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ

**Synthetic Dataset Generation**

In [14]:
def generate_synthetic_dataset(num_samples=1000, num_features=20, seed=42):
    """
    Generates a synthetic binary classification dataset for deep learning testing.
    Returns X of shape (num_features, num_samples) and Y of shape (1, num_samples).
    """
    np.random.seed(seed)
    # Generate random features and build a non-linear decision boundary
    X = np.random.randn(num_features, num_samples)
    true_weights = np.random.randn(1, num_features)
    logits = np.dot(true_weights, X) + np.sin(X[0, :]) * 2.0
    Y = (logits > 0).astype(int)
    
    return X, Y

# Generate dataset for model execution
X_data, Y_data = generate_synthetic_dataset()
print(f"Dataset generated  Features shape: {X_data.shape}, Labels shape: {Y_data.shape}")

Dataset generated  Features shape: (20, 1000), Labels shape: (1, 1000)


**Shallow Network Parameter Initialization (2-Layer Network)**

In [15]:
def initialize_parameters(n_x, n_h, n_y):
    """
    Initializes weight matrices and bias vectors for a 2-layer neural network.
    
    Parameters:
    n_x -- size of the input layer
    n_h -- size of the hidden layer
    n_y -- size of the output layer
    """
    np.random.seed(1)
    
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    
    parameters = {
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2
    }
    
    return parameters

**Deep Network Parameter Initialization (L-Layer Network)**

In [16]:
def initialize_parameters_deep(layer_dims):
    """
    Initializes parameters for an L-layer deep neural network using He-style scaling.
    
    Parameters:
    layer_dims -- python array (list) containing the dimensions of each layer in the network
    """
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2 / layer_dims[l-1])
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
        
    return parameters

In [24]:

t_layers_dims = [20, 5, 3, 1]
t_params_deep = initialize_parameters_deep(t_layers_dims)

print("--- Cell 4 Test ---")
for key in sorted(t_params_deep.keys()):
    print(f"{key} shape: {t_params_deep[key].shape}")

--- Cell 4 Test ---
W1 shape: (5, 20)
W2 shape: (3, 5)
W3 shape: (1, 3)
b1 shape: (5, 1)
b2 shape: (3, 1)
b3 shape: (1, 1)


**Forward Propagation Linear Module**

In [25]:
def linear_forward(A, W, b):
    """
    Computes the linear component of a layer's forward propagation: Z = W * A + b.
    
    Parameters:
    A -- activations from previous layer (or input data)
    W -- weights matrix
    b -- bias vector
    """
    Z = np.dot(W, A) + b
    cache = (A, W, b)
    
    return Z, cache

**Linear-Activation Forward Step**

In [26]:
def linear_activation_forward(A_prev, W, b, activation):
    """
    Computes forward propagation for the Linear -> Activation block.
    
    Parameters:
    A_prev -- activations from previous layer
    W -- weights matrix
    b -- bias vector
    activation -- activation choice: "sigmoid" or "relu"
    """
    if activation == "sigmoid":
        Z, linear_cache = linear_forward(A_prev, W, b)
        A, activation_cache = sigmoid(Z)
    elif activation == "relu":
        Z, linear_cache = linear_forward(A_prev, W, b)
        A, activation_cache = relu(Z)
        
    cache = (linear_cache, activation_cache)
    return A, cache

**Full L-Layer Forward Propagation Model**

In [27]:
def L_model_forward(X, parameters):
    """
    Executes forward propagation for the [LINEAR->RELU]*(L-1) -> LINEAR->SIGMOID network.
    
    Parameters:
    X -- input data matrix of shape (input_size, number_of_examples)
    parameters -- dictionary containing initialized weights and biases
    """
    caches = []
    A = X
    L = len(parameters) // 2
    
    # Forward pass for hidden layers with ReLU
    for l in range(1, L):
        A_prev = A 
        A, cache = linear_activation_forward(
            A_prev, 
            parameters['W' + str(l)], 
            parameters['b' + str(l)], 
            activation="relu"
        )
        caches.append(cache)
        
    # Forward pass for output layer with Sigmoid
    AL, cache = linear_activation_forward(
        A, 
        parameters['W' + str(L)], 
        parameters['b' + str(L)], 
        activation="sigmoid"
    )
    caches.append(cache)
    
    return AL, caches

In [28]:

t_AL, t_caches = L_model_forward(X_data, t_params_deep)

print("--- Cell 7 Test ---")
print("Final Output AL Shape: ", t_AL.shape)
print("Total Layers Caches:   ", len(t_caches))
print("Sample Predictions (AL):", t_AL[0, :5])

--- Cell 7 Test ---
Final Output AL Shape:  (1, 1000)
Total Layers Caches:    3
Sample Predictions (AL): [0.59526467 0.68570752 0.50192518 0.50851243 0.55693848]


**Cross-Entropy Cost Function**

In [20]:
def compute_cost(AL, Y):
    """
    Computes the binary cross-entropy cost function.
    
    Parameters:
    AL -- probability vector corresponding to label predictions, shape (1, number of examples)
    Y -- true "label" vector, shape (1, number of examples)
    """
    m = Y.shape[1]
    
    # Compute cross-entropy loss with stability tolerance
    cost = -1/m * np.sum(
        np.multiply(Y, np.log(AL + 1e-15)) + 
        np.multiply(1 - Y, np.log(1 - AL + 1e-15))
    )
    
    cost = np.squeeze(cost)
    return cost

In [29]:
t_cost = compute_cost(t_AL, Y_data)

print("--- Cell 8 Test ---")
print("Computed Binary Cross-Entropy Cost:", t_cost)

--- Cell 8 Test ---
Computed Binary Cross-Entropy Cost: 0.7371792027758508


**Backward Propagation Pipeline**

In [21]:
def linear_backward(dZ, cache):
    """
    Computes the linear portion of backward propagation for a single layer.
    """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    dW = (1 / m) * np.dot(dZ, A_prev.T)
    db = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(W.T, dZ)
    
    return dA_prev, dW, db

def linear_activation_backward(dA, cache, activation):
    """
    Computes backward propagation for the Linear -> Activation block.
    """
    linear_cache, activation_cache = cache
    
    if activation == "relu":
        dZ = relu_backward(dA, activation_cache)
        dA_prev, dW, db = linear_backward(dZ, linear_cache)
    elif activation == "sigmoid":
        dZ = sigmoid_backward(dA, activation_cache)
        dA_prev, dW, db = linear_backward(dZ, linear_cache)
        
    return dA_prev, dW, db

def L_model_backward(AL, Y, caches):
    """
    Executes backward propagation for the complete L-layer neural network.
    """
    grads = {}
    L = len(caches)
    m = AL.shape[1]
    Y = Y.reshape(AL.shape)
    
    # Initializing backpropagation with output layer derivative
    dAL = - (np.divide(Y, AL + 1e-15) - np.divide(1 - Y, 1 - AL + 1e-15))
    
    # Lth layer (SIGMOID -> LINEAR) gradients
    current_cache = caches[L - 1]
    grads["dA" + str(L - 1)], grads["dW" + str(L)], grads["db" + str(L)] = linear_activation_backward(
        dAL, current_cache, activation="sigmoid"
    )
    
    # Loop backwards through hidden layers (RELU -> LINEAR)
    for l in reversed(range(L - 1)):
        current_cache = caches[l]
        dA_prev_temp, dW_temp, db_temp = linear_activation_backward(
            grads["dA" + str(l + 1)], current_cache, activation="relu"
        )
        grads["dA" + str(l)] = dA_prev_temp
        grads["dW" + str(l + 1)] = dW_temp
        grads["db" + str(l + 1)] = db_temp

    return grads

In [31]:
t_grads = L_model_backward(t_AL, Y_data, t_caches)

print("Cell 9 Test ")
for key in sorted(t_grads.keys()):
    print(f"{key} shape: {t_grads[key].shape}")

Cell 9 Test 
dA0 shape: (20, 1000)
dA1 shape: (5, 1000)
dA2 shape: (3, 1000)
dW1 shape: (5, 20)
dW2 shape: (3, 5)
dW3 shape: (1, 3)
db1 shape: (5, 1)
db2 shape: (3, 1)
db3 shape: (1, 1)


**Parameter Update & Full Training Pipeline Test**

In [22]:
def update_parameters(params, grads, learning_rate):
    """
    Updates parameters using gradient descent optimization.
    """
    parameters = params.copy()
    L = len(parameters) // 2

    for l in range(L):
        parameters["W" + str(l + 1)] -= learning_rate * grads["dW" + str(l + 1)]
        parameters["b" + str(l + 1)] -= learning_rate * grads["db" + str(l + 1)]
        
    return parameters

def L_layer_model(X, Y, layers_dims, learning_rate=0.0075, num_iterations=2500, print_cost=False):
    """
    Implements an L-layer neural network model training loop.
    """
    np.random.seed(1)
    costs = []
    
    # Initialize parameters
    parameters = initialize_parameters_deep(layers_dims)
    
    # Loop over training iterations
    for i in range(0, num_iterations):
        # Forward propagation
        AL, caches = L_model_forward(X, parameters)
        
        # Compute cost
        cost = compute_cost(AL, Y)
        
        # Backward propagation
        grads = L_model_backward(AL, Y, caches)
        
        # Update parameters
        parameters = update_parameters(parameters, grads, learning_rate)
        
        # Print loss every 500 iterations
        if print_cost and i % 500 == 0:
            print(f"Cost after iteration {i}: {cost:.6f}")
            costs.append(cost)
            
    return parameters, costs

# Execution test using synthetic dataset from Cell 2
if __name__ == "__main__":
    # Define a 4-layer architecture: 20 input units -> 7 hidden -> 4 hidden -> 1 output
    network_architecture = [20, 7, 4, 1]
    
    print("Training Deep Neural Network...")
    trained_parameters, cost_history = L_layer_model(
        X_data, 
        Y_data, 
        network_architecture, 
        learning_rate=0.01, 
        num_iterations=2000, 
        print_cost=True
    )

   

Training Deep Neural Network...
Cost after iteration 0: 0.854470
Cost after iteration 500: 0.650926
Cost after iteration 1000: 0.479468
Cost after iteration 1500: 0.220179
